# Unified Model Benchmark

Runs all five available risk models over the `calibration` and `validation`
datasets, producing probability-map rasters for each, then a side-by-side
visual comparison.

Models split into two families with different `fit`/`apply` interfaces:

| Family | Models | `fit()` | `apply()` |
|--------|--------|---------|-----------|
| ML | `GLMModel`, `RFModel`, `ICARModel` | uses attached `dataset` + `sampling` | `apply(output_file, dataset, mask, mask_value)` |
| Benchmark | `MWModel`, `JNRBenchmarkModel` | `fit(dataset=, …, folder=)` | `apply(dataset=, output_folder=, time_interval=)` → dict{window→Path} |

Each model is fit on **calibration** and applied to **both** periods.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")

from spatialrisk import (
    Project,
    Dataset,
    GLMModel,
    RFModel,
    ICARModel,
    MWModel,
    JNRBenchmarkModel,
    rmj,
)
from spatialrisk.sampling import Sampling, SamplingStrategy

In [3]:
project_name = "mtq-refactor"
project = Project.load(project_name=project_name)
project.list_datasets()

Loaded 3 model(s)
✓ Target set: forest_loss_2015_2020 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2015): forest_gfc, forest_gfc_edge, towns_dist
✓ Target set: forest_loss_2020_2024 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2020): forest_gfc, forest_gfc_edge, towns_dist
Loaded 2 dataset(s)
Project loaded from: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
Loaded 23 processed variables


['calibration', 'validation']

In [4]:
calibration = project.get_dataset("calibration")
validation = project.get_dataset("validation")
calibration.validate()


🔍 Validating dataset configuration...
✓ Checking variable existence...
✓ All variables exist
✓ All variables processed and available
✓ Checking spatial compatibility...

✅ Dataset validation passed!


True

### Shared parameters

`periods` maps each dataset to its time interval in years
(calibration = 2020 − 2015 = 5, validation = 2024 − 2020 = 4).
`forest_mask` is the forest raster used to mask ML-model predictions.
`results` accumulates one representative probability raster per
`(model, period)` for the comparison grid at the end.

In [5]:
random_seed = 1

sampling = Sampling(
    strategy=SamplingStrategy.legacy,
    n_samples=10000,
    seed=33,            # for reproducibility
    adapt=True,
    pixel_area_ha=0.09,
)

# period name -> (dataset, time_interval_years)
periods = {
    "calibration": (calibration, 5),
    "validation": (validation, 4),
}

# Forest raster used as the mask for ML-model apply()
forest_mask = calibration.get_file_paths()["forest_gfc"]

# (model_name, period_name) -> Path to a single representative probability raster
results = {}

## 2. ML models

### 2.1 GLM — logistic regression

Constructed with the calibration dataset + sampling attached; `fit()` trains on
calibration, then `apply()` predicts a probability raster for each period,
masked to forest (`mask_value=0`).

In [6]:
glm = GLMModel(
    name="glm_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
glm.register(project)
glm.fit()

  Model registered as project.models['glm_glm_v1']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json

🔍 Validating dataset configuration...
✓ Checking variable existence...
✓ All variables exist
✓ All variables processed and available
✓ Checking spatial compatibility...

✅ Dataset validation passed!

📊 Creating DataFrame with SamplingStrategy.legacy sampling...
  Valid pixels: 846,547
  Adapted n_samples to 10,000 (total area: 0.076 Mha)
  Sampled 3,363 deforested + 10,000 forest = 13,363 total pixels (legacy)
✓ DataFrame created: 13,363 rows × 12 columns
  Target column: 'target' (from 'forest_loss_2015_2020')
  Samples saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/samples_glm_glm_v1.csv

📊 Generating Patsy formula:
  Target: forest_loss_2015_2020
  Features: altitude, forest_gfc, forest_gfc_edge, protected_area, rivers_dist, roads_dist, slope, subj, to

GLMModel(name='glm_v1', model_type='glm', project_name='mtq-refactor', dataset_name=None, target_name='forest_loss_2015_2020', feature_names=['altitude', 'forest_gfc', 'forest_gfc_edge', 'protected_area', 'rivers_dist', 'roads_dist', 'slope', 'subj', 'towns_dist'], year=2015, formula='I(forest_loss_2015_2020) + trial ~ scale(altitude) + scale(forest_gfc_edge) + scale(rivers_dist) + scale(roads_dist) + scale(slope) + scale(towns_dist) + C(forest_gfc, levels=[0, 1]) + C(protected_area, levels=[0]) + C(subj, levels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34])', parameters={}, sampling=Sampling(strategy=<SamplingStrategy.legacy: 'legacy'>, n_samples=10000, seed=33, adapt=True, pixel_area_ha=0.09), model_path=PosixPath('/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_glm_v1_20260616_155340.pickle'), samples_path=PosixPath('/home/jserafini/Desktop/projects/spati

In [7]:
for period, (ds, _ti) in periods.items():
    out = project.folders.glm_model / glm.name / f"{period}.tif"
    glm.apply(out, ds, forest_mask, 0)
    results[("GLM", period)] = out
    print(f"GLM {period:12s} → {out}")


🗺  Predicting GLM raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_v1/calibration.tif
✓ GLM raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_v1/calibration.tif
  Prediction registered as project.predictions['glm_glm_v1__calibration_y2015']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
GLM calibration  → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_v1/calibration.tif

🗺  Predicting GLM raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_v1/validation.tif
✓ GLM raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_glm/glm_v1/validation.tif
  Prediction registered as project.predictions['glm_glm_v1__validation_y2015']
Project saved to: /ho

### 2.2 Random Forest

In [8]:
rf = RFModel(
    name="rf_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
rf.register(project)
rf.fit()

  Model registered as project.models['rf_rf_v1']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json

🔍 Validating dataset configuration...
✓ Checking variable existence...
✓ All variables exist
✓ All variables processed and available
✓ Checking spatial compatibility...

✅ Dataset validation passed!

📊 Creating DataFrame with SamplingStrategy.legacy sampling...
  Valid pixels: 846,547
  Adapted n_samples to 10,000 (total area: 0.076 Mha)
  Sampled 3,363 deforested + 10,000 forest = 13,363 total pixels (legacy)
✓ DataFrame created: 13,363 rows × 12 columns
  Target column: 'target' (from 'forest_loss_2015_2020')
  Samples saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/samples_rf_rf_v1.csv

📊 Generating Patsy formula:
  Target: forest_loss_2015_2020
  Features: altitude, forest_gfc, forest_gfc_edge, protected_area, rivers_dist, roads_dist, slope, subj, towns_d

RFModel(name='rf_v1', model_type='rf', project_name='mtq-refactor', dataset_name=None, target_name='forest_loss_2015_2020', feature_names=['altitude', 'forest_gfc', 'forest_gfc_edge', 'protected_area', 'rivers_dist', 'roads_dist', 'slope', 'subj', 'towns_dist'], year=2015, formula='I(forest_loss_2015_2020) + trial ~ scale(altitude) + scale(forest_gfc_edge) + scale(rivers_dist) + scale(roads_dist) + scale(slope) + scale(towns_dist) + C(forest_gfc, levels=[0, 1]) + C(protected_area, levels=[0]) + C(subj, levels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34])', parameters={}, sampling=Sampling(strategy=<SamplingStrategy.legacy: 'legacy'>, n_samples=10000, seed=33, adapt=True, pixel_area_ha=0.09), model_path=PosixPath('/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_rf_v1_20260616_155348.pickle'), samples_path=PosixPath('/home/jserafini/Desktop/projects/spatial_ris

In [9]:
for period, (ds, _ti) in periods.items():
    out = project.folders.rf_model / rf.name / f"{period}.tif"
    rf.apply(out, ds, forest_mask, 0)
    results[("RF", period)] = out
    print(f"RF  {period:12s} → {out}")


🗺  Predicting RF raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_v1/calibration.tif
✓ RF raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_v1/calibration.tif
  Prediction registered as project.predictions['rf_rf_v1__calibration_y2015']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
RF  calibration  → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_v1/calibration.tif

🗺  Predicting RF raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_v1/validation.tif
✓ RF raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_rf/rf_v1/validation.tif
  Prediction registered as project.predictions['rf_rf_v1__validation_y2015']
Project saved to: /home/jserafini/Deskt

### 2.3 iCAR — spatial logistic regression (MCMC)

In [10]:
icar = ICARModel(
    name="icar_v1",
    dataset=calibration,
    sampling=sampling,
    random_seed=random_seed,
)
icar.register(project)
icar.fit()

  Model registered as project.models['icar_icar_v1']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json

🔍 Validating dataset configuration...
✓ Checking variable existence...
✓ All variables exist
✓ All variables processed and available
✓ Checking spatial compatibility...

✅ Dataset validation passed!

📊 Creating DataFrame with SamplingStrategy.legacy sampling...
  Valid pixels: 846,547
  Adapted n_samples to 10,000 (total area: 0.076 Mha)
  Sampled 3,363 deforested + 10,000 forest = 13,363 total pixels (legacy)
✓ DataFrame created: 13,363 rows × 12 columns
  Target column: 'target' (from 'forest_loss_2015_2020')
  Samples saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/samples_icar_icar_v1.csv

📊 Generating Patsy formula:
  Target: forest_loss_2015_2020
  Features: altitude, forest_gfc, forest_gfc_edge, protected_area, rivers_dist, roads_dist, slope, sub

ICARModel(name='icar_v1', model_type='icar', project_name='mtq-refactor', dataset_name=None, target_name='forest_loss_2015_2020', feature_names=['altitude', 'forest_gfc', 'forest_gfc_edge', 'protected_area', 'rivers_dist', 'roads_dist', 'slope', 'subj', 'towns_dist'], year=2015, formula='I(forest_loss_2015_2020) + trial ~ scale(altitude) + scale(forest_gfc_edge) + scale(rivers_dist) + scale(roads_dist) + scale(slope) + scale(towns_dist) + C(forest_gfc, levels=[0, 1]) + C(protected_area, levels=[0]) + C(subj, levels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34])', parameters={}, sampling=Sampling(strategy=<SamplingStrategy.legacy: 'legacy'>, n_samples=10000, seed=33, adapt=True, pixel_area_ha=0.09), model_path=PosixPath('/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_icar_v1_20260616_160037.pickle'), samples_path=PosixPath('/home/jserafini/Desktop/projects

In [11]:
for period, (ds, _ti) in periods.items():
    out = project.folders.icar_model / icar.name / f"{period}.tif"
    icar.apply(out, ds, forest_mask, 0)
    results[("ICAR", period)] = out
    print(f"ICAR {period:11s} → {out}")


🗺  Predicting iCAR raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_v1/calibration.tif
✓ iCAR raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_v1/calibration.tif
  Prediction registered as project.predictions['icar_icar_v1__calibration_y2015']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
ICAR calibration → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_v1/calibration.tif

🗺  Predicting iCAR raster → /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_v1/validation.tif
✓ iCAR raster written: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/far_icar/icar_v1/validation.tif
  Prediction registered as project.predictions['icar_icar_v1__validation_y2015']
Pro

## 3. Benchmark models

### 3.1 Moving Window (MW)

Fit on calibration, then applied to each period. `apply()` returns a dict
mapping each window size to a probability raster; we keep window **11** as the
representative map for the comparison grid.

In [12]:
WIN_REPR = 11  # representative window size for the comparison grid

mw = MWModel(
    name="calibration_mw",
    forest_edge_var="forest_gfc_edge",
    forest_var="forest_gfc",
    win_size_list=[5, 11, 21],
    defor_threshold=99.5,
    max_dist=50000,
    blk_rows=256,
)
mw.fit(dataset=calibration, time_interval=5, folder=project.folders.rmj_mw)
mw.register(project)
print(f"dist_thresh   : {mw.dist_thresh:.1f} m")
print(f"ldefrate_files: {list(mw.ldefrate_files.keys())} window sizes")


🔧 MW fit — period='calibration', windows=[5, 11, 21]
  dist_thresh=270.0 m
  local_defor_rate — window 5×5 px...
  local_defor_rate — window 11×11 px...
  local_defor_rate — window 21×21 px...
✓ MW fit complete — 3 ldefrate files, trained_at=2026-06-16T16:00:44.947702
  Model registered as project.models['mw_calibration_mw']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
dist_thresh   : 270.0 m
ldefrate_files: ['5', '11', '21'] window sizes


In [13]:
for period, (ds, ti) in periods.items():
    outputs = mw.apply(
        dataset=ds,
        time_interval=ti,
        output_folder=project.folders.rmj_mw,
    )
    results[("MW", period)] = outputs[str(WIN_REPR)]  # MW keys are strings
    print(f"MW  {period:12s} (win {WIN_REPR}) → {outputs[str(WIN_REPR)].name}")


🗺  MW apply — period='calibration', windows=[5, 11, 21]
  Prediction registered as project.predictions['mw_calibration_mw__calibration_y2015_w5']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
  window 5 → prob_mw_5_calibration.tif
  Prediction registered as project.predictions['mw_calibration_mw__calibration_y2015_w11']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
  window 11 → prob_mw_11_calibration.tif
  Prediction registered as project.predictions['mw_calibration_mw__calibration_y2015_w21']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
  window 21 → prob_mw_21_calibration.tif
✓ MW apply complete — 3 probability maps written
MW  calibration  (win 11) → prob_mw_11_calibration.tif

🗺  MW apply — period='validation', windows=[5, 11, 21]
  Pred

### 3.2 JNR benchmark

Unlike MW, JNR's `apply()` writes a **single** vulnerability raster per period
and returns its `Path`. The validation period reuses the calibration period's
deforestation-rate table (`deforate_model=`), so calibration must run first.

In [14]:
jnr = JNRBenchmarkModel(
    name="calibration_jnr",
    forest_edge_var="forest_gfc_edge",
    forest_var="forest_gfc",
    subj_var="subj",
    defor_threshold=99.5,
    max_dist=50000,
    blk_rows=128,
)
jnr.fit(dataset=calibration, defor_threshold=99.5, folder=project.folders.rmj_bm)
jnr.register(project)
print(f"dist_thresh : {jnr.dist_thresh:.1f} m")
print(f"dist_bins   : {len(jnr.dist_bins)} edges → {len(jnr.dist_bins) - 1} classes")


🔧 JNR fit — period='calibration'
/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data/forest_loss_2015_2020_reprojected_matched.tif /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data/forest_gfc_reprojected_matched_edge_2015.tif
  dist_thresh=270.0 m
  dist_bins: 30 edges
✓ JNR fit complete — trained_at=2026-06-16T16:00:47.435641
  Model registered as project.models['jnr_calibration_jnr']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
dist_thresh : 270.0 m
dist_bins   : 30 edges → 29 classes


In [15]:
# Calibration must run before validation: validation reuses the calibration
# defrate table for quantity-adjustment.
for period, (ds, ti) in periods.items():
    out = project.folders.rmj_bm / period / f"prob_bm_{period}.tif"
    deforate_model = (
        None if period == "calibration" else jnr.defrate_files.get("calibration")
    )
    jnr.apply(
        output_file=out,
        dataset=ds,
        time_interval=ti,
        deforate_model=deforate_model,
    )
    results[("JNR", period)] = out
    print(f"JNR {period:12s} → {out.name}")


🗺  JNR apply — period='calibration' → prob_bm_calibration.tif
✓ JNR apply complete — /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/rmj_bm/calibration/prob_bm_calibration.tif
  Prediction registered as project.predictions['jnr_calibration_jnr__calibration_y2015']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
JNR calibration  → prob_bm_calibration.tif

🗺  JNR apply — period='validation' → prob_bm_validation.tif
✓ JNR apply complete — /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/rmj_bm/validation/prob_bm_validation.tif
  Prediction registered as project.predictions['jnr_calibration_jnr__validation_y2015']
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
JNR validation   → prob_bm_validation.tif


## 4. Visual comparison

A 5-column (one per model) × 2-row (calibration, validation) grid of the
probability rasters, on a shared colormap and color scale. The helper reads
each raster, masks its nodata value (ML rasters use nodata=0, benchmark rasters
use 65535), and normalizes to [0, 1] for display.

In [16]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt


def load_prob(path):
    """Read a probability raster, mask nodata, normalize to [0, 1]."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata
    # Mask known nodata conventions: 0 (ML) and 65535 (benchmark)
    mask = np.isnan(arr)
    if nodata is not None:
        mask |= arr == nodata
    mask |= arr == 0
    mask |= arr == 65535
    arr = np.ma.masked_array(arr, mask=mask)
    vmax = arr.max()
    if vmax and vmax > 0:
        arr = arr / vmax
    return arr

In [17]:
model_order = ["GLM", "RF", "ICAR", "MW", "JNR"]
period_order = ["calibration", "validation"]

fig, axes = plt.subplots(
    len(period_order),
    len(model_order),
    figsize=(4 * len(model_order), 4 * len(period_order)),
)
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("lightgrey")

for r, period in enumerate(period_order):
    for c, model_name in enumerate(model_order):
        ax = axes[r, c]
        ax.set_xticks([])
        ax.set_yticks([])
        path = results.get((model_name, period))
        if path is None:
            ax.text(0.5, 0.5, "no output", ha="center", va="center")
            continue
        im = ax.imshow(load_prob(path), cmap=cmap, vmin=0, vmax=1)
        if r == 0:
            ax.set_title(model_name, fontsize=13)
        if c == 0:
            ax.set_ylabel(period, fontsize=12)

fig.colorbar(im, ax=axes, shrink=0.6, label="relative risk (normalized)")
fig.suptitle("Model probability maps — mtq-refactor", fontsize=15, y=0.98)
plt.show()

/tmp/ipykernel_2187416/3086699803.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
print(project.list_predictions())

['glm_glm_v1__calibration_y2015', 'glm_glm_v1__validation_y2015', 'rf_rf_v1__calibration_y2015', 'rf_rf_v1__validation_y2015', 'icar_icar_v1__calibration_y2015', 'icar_icar_v1__validation_y2015', 'mw_calibration_mw__calibration_y2015_w5', 'mw_calibration_mw__calibration_y2015_w11', 'mw_calibration_mw__calibration_y2015_w21', 'mw_calibration_mw__validation_y2015_w5', 'mw_calibration_mw__validation_y2015_w11', 'mw_calibration_mw__validation_y2015_w21', 'jnr_calibration_jnr__calibration_y2015', 'jnr_calibration_jnr__validation_y2015']
